# Task 3.2

This notebook answers all parts of Task 3.2 from the project description. It contains the MILP formulation, the relaxation, a brute-force solution approach, and a runnable implementation based on the dataset.


In [1]:
from brew_utils import *
from scipy.optimize import linprog

## 3.2.1 Decision Variables and Data

This section prepares the data and defines the decision vector used in the model.

The decision variables are:
- `b_i`: kilograms of barley type `i`
- `m_j`: kilograms of malt type `j`
- `e_k`: kilograms of malt extract type `k`
- `h_l`: kilograms of hop type `l`
- `y_r`: kilograms of yeast type `r`
- `z_malt`: binary variable equal to 1 if the malting process is used
- `z_mash`: binary variable equal to 1 if the mashing process is used

The code below converts the ingredient tables into NumPy arrays and sets up the index ranges for these variables inside one optimization vector. This is the first part of the MILP formulation requested in 3.2.1.


In [2]:
# Data arrays
barley_cost = barley["cost"].to_numpy(dtype=float)
barley_ebc = barley["EBC"].to_numpy(dtype=float)
barley_q = barley["quality"].to_numpy(dtype=float)

malt_cost = malts["cost"].to_numpy(dtype=float)
malt_ebc = malts["EBC"].to_numpy(dtype=float)
malt_q = malts["quality"].to_numpy(dtype=float)

extract_cost = maltextracts["cost"].to_numpy(dtype=float)
extract_ebc = maltextracts["EBC"].to_numpy(dtype=float)
extract_q = maltextracts["quality"].to_numpy(dtype=float)

hop_cost = hops["cost"].to_numpy(dtype=float)
hop_q = hops["quality"].to_numpy(dtype=float)

yeast_cost = yeasts["cost"].to_numpy(dtype=float)
yeast_q = yeasts["quality"].to_numpy(dtype=float)

F_malt = float(maltprocess.loc[0, "fixedcost"])
v_malt = float(maltprocess.loc[0, "variablecost"])

F_mash = float(mashingprocess.loc[0, "fixedcost"])
v_mash = float(mashingprocess.loc[0, "variablecost"])

nB = len(barley)
nM = len(malts)
nE = len(maltextracts)
nH = len(hops)
nY = len(yeasts)

iB0 = 0
iM0 = iB0 + nB
iE0 = iM0 + nM
iH0 = iE0 + nE
iY0 = iH0 + nH
iZm = iY0 + nY
iZs = iZm + 1
nvars = iZs + 1

def slice_B():
    return slice(iB0, iB0 + nB)

def slice_M():
    return slice(iM0, iM0 + nM)

def slice_E():
    return slice(iE0, iE0 + nE)

def slice_H():
    return slice(iH0, iH0 + nH)

def slice_Y():
    return slice(iY0, iY0 + nY)

## 3.2.1 MILP Formulation

This block builds the optimization model. It answers Task 3.2.1 by encoding the objective function and the constraints.

Objective:
- minimize ingredient costs
- add variable malting and mashing costs
- add fixed malting and mashing costs through `z_malt` and `z_mash`

Constraints:
- malt-extract-equivalent balance so the beer volume target is met
- exact hop amount and exact yeast amount
- lower and upper EBC bounds
- minimum quality requirements for malt sources, hops, and yeast
- activation constraints linking process use to the binary variables
- `z_malt <= z_mash`, because malting implies that mashing is also needed later

The parameter `z_fix` is used later for brute force: when it is set, the binary process variables are fixed to a chosen on or off pattern.


In [3]:
def build_lp_model(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast, z_fix=None):
    V_wort = wort_from_beer_volume(V_beer)
    m_tot = malt_extract_eq_needed_for_beer(V_beer)
    H_tot = hops_needed_for_beer(V_beer)
    Y_tot = yeast_needed_for_beer(V_beer)

    gamma = (SG - 1.0) / 0.0344

    M_B = m_tot / BARLEY_TO_MALTEXTRACT_EQ
    M_M = m_tot / MALT_TO_MALTEXTRACT_EQ

    c = np.zeros(nvars)
    c[slice_B()] = barley_cost
    c[slice_M()] = malt_cost
    c[slice_E()] = extract_cost
    c[slice_H()] = hop_cost
    c[slice_Y()] = yeast_cost

    c[slice_B()] += v_malt
    c[slice_B()] += v_mash * BARLEY_TO_MALT
    c[slice_M()] += v_mash

    c[iZm] = F_malt
    c[iZs] = F_mash

    A_eq = []
    b_eq = []

    row = np.zeros(nvars)
    row[slice_B()] = BARLEY_TO_MALTEXTRACT_EQ
    row[slice_M()] = MALT_TO_MALTEXTRACT_EQ
    row[slice_E()] = 1.0
    A_eq.append(row)
    b_eq.append(m_tot)

    row = np.zeros(nvars)
    row[slice_H()] = 1.0
    A_eq.append(row)
    b_eq.append(H_tot)

    row = np.zeros(nvars)
    row[slice_Y()] = 1.0
    A_eq.append(row)
    b_eq.append(Y_tot)

    A_eq = np.array(A_eq, dtype=float)
    b_eq = np.array(b_eq, dtype=float)

    A_ub = []
    b_ub = []

    row = np.zeros(nvars)
    row[slice_B()] = gamma * BARLEY_TO_MALTEXTRACT_EQ * barley_ebc
    row[slice_M()] = gamma * MALT_TO_MALTEXTRACT_EQ * malt_ebc
    row[slice_E()] = gamma * extract_ebc
    A_ub.append(row)
    b_ub.append(EBC_max * m_tot)

    row = np.zeros(nvars)
    row[slice_B()] = -gamma * BARLEY_TO_MALTEXTRACT_EQ * barley_ebc
    row[slice_M()] = -gamma * MALT_TO_MALTEXTRACT_EQ * malt_ebc
    row[slice_E()] = -gamma * extract_ebc
    A_ub.append(row)
    b_ub.append(-EBC_min * m_tot)

    row = np.zeros(nvars)
    row[slice_B()] = -BARLEY_TO_MALTEXTRACT_EQ * barley_q
    row[slice_M()] = -MALT_TO_MALTEXTRACT_EQ * malt_q
    row[slice_E()] = -extract_q
    A_ub.append(row)
    b_ub.append(-Q_malt * m_tot)

    row = np.zeros(nvars)
    row[slice_H()] = -hop_q
    A_ub.append(row)
    b_ub.append(-Q_hop * H_tot)

    row = np.zeros(nvars)
    row[slice_Y()] = -yeast_q
    A_ub.append(row)
    b_ub.append(-Q_yeast * Y_tot)

    row = np.zeros(nvars)
    row[slice_B()] = 1.0
    row[iZm] = -M_B
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[slice_B()] = BARLEY_TO_MALT
    row[slice_M()] = 1.0
    row[iZs] = -M_M
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[iZm] = 1.0
    row[iZs] = -1.0
    A_ub.append(row)
    b_ub.append(0.0)

    A_ub = np.array(A_ub, dtype=float)
    b_ub = np.array(b_ub, dtype=float)

    bounds = [(0, None)] * nvars
    if z_fix is None:
        bounds[iZm] = (0, 1)
        bounds[iZs] = (0, 1)
    else:
        z_malt_fix, z_mash_fix = z_fix
        bounds[iZm] = (z_malt_fix, z_malt_fix)
        bounds[iZs] = (z_mash_fix, z_mash_fix)

    return {
        "c": c,
        "A_eq": A_eq,
        "b_eq": b_eq,
        "A_ub": A_ub,
        "b_ub": b_ub,
        "bounds": bounds,
        "V_wort": V_wort,
        "m_tot": m_tot,
        "H_tot": H_tot,
        "Y_tot": Y_tot,
    }

## 3.2.2 Relaxation and 3.2.3 Brute Force

This section answers parts 3.2.2 and 3.2.3.

For 3.2.2, the relaxation is obtained by allowing `z_malt` and `z_mash` to vary continuously between 0 and 1 instead of forcing them to be binary. This is done by keeping their bounds as `[0, 1]`.

For 3.2.3, brute force is implemented by testing all valid process configurations:
- `(0, 0)`: no malting and no mashing, so only malt extract can be used
- `(0, 1)`: mashing only, so malt and malt extract can be used
- `(1, 1)`: both malting and mashing, so barley, malt, and malt extract can be used

For each case, the binary variables are fixed and the remaining problem is solved as a linear program with `linprog`. The cheapest feasible case is then the MILP solution found by brute force.


In [8]:
def solve_relaxation(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast):
    model = build_lp_model(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast, z_fix=None)

    res = linprog(
        c=model["c"],
        A_ub=model["A_ub"],
        b_ub=model["b_ub"],
        A_eq=model["A_eq"],
        b_eq=model["b_eq"],
        bounds=model["bounds"],
        method="highs",
    )
    return model, res

def solve_bruteforce(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast):
    candidates = [(0, 0), (0, 1), (1, 1)]
    best = None
    all_results = []

    for z_fix in candidates:
        model = build_lp_model(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast, z_fix=z_fix)

        res = linprog(
            c=model["c"],
            A_ub=model["A_ub"],
            b_ub=model["b_ub"],
            A_eq=model["A_eq"],
            b_eq=model["b_eq"],
            bounds=model["bounds"],
            method="highs",
        )

        info = {
            "z_malt": z_fix[0],
            "z_mash": z_fix[1],
            "success": res.success,
            "message": res.message,
            "cost": res.fun if res.success else np.nan,
            "result": res,
            "model": model,
        }
        all_results.append(info)

        if res.success:
            if best is None or res.fun < best["cost"]:
                best = info

    return best, all_results

## 3.2.1 and 3.2.4 Output Interpretation

These helper functions turn the solver output into a readable answer for the report.

They compute:
- the non-zero decision variables
- the resulting EBC value
- the resulting average qualities
- the total cost and process choices

This is useful for Task 3.2.4, where the program must not only solve the model but also present the solution clearly.


In [9]:
def solution_to_dataframe(x, tol=1e-9):
    names = (
        list(barley["name"])
        + list(malts["name"])
        + list(maltextracts["name"])
        + list(hops["name"])
        + list(yeasts["name"])
        + ["z_malt", "z_mash"]
    )

    df = pd.DataFrame({"variable": names, "value": x})
    return df[df["value"].abs() > tol].reset_index(drop=True)

def summarize_solution(model, res):
    x = res.x

    b = x[slice_B()]
    m = x[slice_M()]
    e = x[slice_E()]
    h = x[slice_H()]
    y = x[slice_Y()]

    m_tot = model["m_tot"]
    H_tot = model["H_tot"]
    Y_tot = model["Y_tot"]

    gamma = (SG - 1.0) / 0.0344

    ebc_value = gamma * (
        BARLEY_TO_MALTEXTRACT_EQ * np.sum(barley_ebc * b)
        + MALT_TO_MALTEXTRACT_EQ * np.sum(malt_ebc * m)
        + np.sum(extract_ebc * e)
    ) / m_tot if m_tot > 0 else 0.0

    q_malt_value = (
        BARLEY_TO_MALTEXTRACT_EQ * np.sum(barley_q * b)
        + MALT_TO_MALTEXTRACT_EQ * np.sum(malt_q * m)
        + np.sum(extract_q * e)
    ) / m_tot if m_tot > 0 else 0.0

    q_hop_value = np.sum(hop_q * h) / H_tot if H_tot > 0 else 0.0
    q_yeast_value = np.sum(yeast_q * y) / Y_tot if Y_tot > 0 else 0.0

    return {
        "optimal_cost": res.fun,
        "success": res.success,
        "message": res.message,
        "V_wort": model["V_wort"],
        "m_tot": m_tot,
        "H_tot": H_tot,
        "Y_tot": Y_tot,
        "EBC": ebc_value,
        "Q_malt": q_malt_value,
        "Q_hop": q_hop_value,
        "Q_yeast": q_yeast_value,
        "z_malt": x[iZm],
        "z_mash": x[iZs],
    }

def print_solution(title, model, res):
    print(f"\n=== {title} ===")
    print("Success:", res.success)
    print("Message:", res.message)

    if not res.success:
        return

    summary = summarize_solution(model, res)
    for k, v in summary.items():
        print(f"{k}: {v}")

    print("\nNon-zero variables:")
    df = solution_to_dataframe(res.x)
    if df.empty:
        print("No non-zero variables")
    else:
        print(df.to_string(index=False))

def run_problem(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast):
    print("Input parameters:")
    print("V_beer =", V_beer)
    print("EBC_min =", EBC_min)
    print("EBC_max =", EBC_max)
    print("Q_malt =", Q_malt)
    print("Q_hop =", Q_hop)
    print("Q_yeast =", Q_yeast)

    model = build_lp_model(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast)
    print("\nModel dimensions:")
    print("c shape:", model["c"].shape)
    print("A_eq shape:", model["A_eq"].shape)
    print("A_ub shape:", model["A_ub"].shape)
    print("Number of bounds:", len(model["bounds"]))

    relax_model, relax_res = solve_relaxation(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast)
    print_solution("Relaxation", relax_model, relax_res)

    best_solution, all_results = solve_bruteforce(V_beer, EBC_min, EBC_max, Q_malt, Q_hop, Q_yeast)

    print("\nAll brute-force cases:")
    all_df = pd.DataFrame([
        {
            "z_malt": r["z_malt"],
            "z_mash": r["z_mash"],
            "success": r["success"],
            "cost": r["cost"],
            "message": r["message"],
        }
        for r in all_results
    ])
    print(all_df.to_string(index=False))

    if best_solution is not None:
        print("\nBest brute-force choice:")
        print((best_solution["z_malt"], best_solution["z_mash"]))
        print_solution("Original MILP via brute force", best_solution["model"], best_solution["result"])
    else:
        print("No feasible solution found for the original MILP.")

    return {
        "relaxation": (relax_model, relax_res),
        "bruteforce_best": best_solution,
        "bruteforce_all": all_results,
    }

## 3.2.4 Model Parameters

This block sets the input parameters required by the assignment.

The program is written with explicit parameters for:
- `V_beer`
- `EBC_min`
- `EBC_max`
- `Q_malt`
- `Q_hop`
- `Q_yeast`

Changing these values lets the same notebook solve other instances of the problem.


In [10]:
V_beer = 5000
EBC_min = 20
EBC_max = 29
Q_malt = 3
Q_hop = 2
Q_yeast = 3

## 3.2.4 Run the Program

This final block runs the complete workflow required in Task 3.2.4:
- build the model
- solve the relaxation
- solve the original MILP by brute force
- print the relevant results in a report-friendly format


In [7]:
results = run_problem(
    V_beer=V_beer,
    EBC_min=EBC_min,
    EBC_max=EBC_max,
    Q_malt=Q_malt,
    Q_hop=Q_hop,
    Q_yeast=Q_yeast,
)

Input parameters:
V_beer = 5000
EBC_min = 20
EBC_max = 29
Q_malt = 3
Q_hop = 2
Q_yeast = 3

Model dimensions:
c shape: (62,)
A_eq shape: (3, 62)
A_ub shape: (8, 62)
Number of bounds: 62

=== Relaxation ===
Success: True
Message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
optimal_cost: 4868.170216649008
success: True
message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
V_wort: 5263.1578947368425
m_tot: 763.1578947368429
H_tot: 6.842105263157895
Y_tot: 3.75
EBC: 20.000000000000004
Q_malt: 3.0
Q_hop: 2.0000000000000004
Q_yeast: 3.0
z_malt: 0.025234899328858976
z_mash: 0.9999999999999999

Non-zero variables:
variable      value
barley03  32.097021
  malt03 905.801837
  malt06  24.072766
   hop07   5.131579
   hop10   1.710526
 yeast03   3.750000
  z_malt   0.025235
  z_mash   1.000000

All brute-force cases:
 z_malt  z_mash  success        cost                                                         message
      0       0     True 7411.833344 Optim